In [1]:
%load_ext sql

In [2]:
%sql sqlite:///library.db

Connecting to 'sqlite:///library.db'

In [3]:
%%sql
CREATE TABLE Department (
    deptID INTEGER PRIMARY KEY AUTOINCREMENT,
    deptName TEXT NOT NULL UNIQUE,
    location TEXT
)

Running query in 'sqlite:///library.db'

In [4]:
%%sql
CREATE TABLE PostalCode (
    postalCode TEXT PRIMARY KEY,
    city TEXT NOT NULL,
    province TEXT NOT NULL
)

Running query in 'sqlite:///library.db'

++
||
++
++

In [5]:
%%sql
CREATE TABLE Member (
    memberID INTEGER PRIMARY KEY AUTOINCREMENT,
    firstName TEXT NOT NULL,
    LastName TEXT NOT NULL,
    email TEXT NOT NULL UNIQUE,
    phone TEXT,
    street TEXT,
    postalCode TEXT,
    registrationDate TEXT NOT NULL DEFAULT (date('now')),
    membershipType TEXT NOT NULL CHECK (MembershipType IN ('Standard','Student','Senior')),
    FOREIGN KEY (postalCode) REFERENCES PostalCode(PostalCode)
    ON UPDATE CASCADE ON DELETE SET NULL
)

Running query in 'sqlite:///library.db'

++
||
++
++

In [6]:
%%sql
CREATE TABLE positionDepartment (
    position TEXT PRIMARY KEY,
    deptID INTEGER NOT NULL,
    FOREIGN KEY (deptID) REFERENCES Department(deptID)
    ON UPDATE CASCADE ON DELETE RESTRICT
)

Running query in 'sqlite:///library.db'

++
||
++
++

In [7]:
%%sql
CREATE TABLE Employee (
    employeeID INTEGER PRIMARY KEY AUTOINCREMENT,
    firstName TEXT NOT NULL,
    lastName TEXT NOT NULL,
    position TEXT NOT NULL,
    hireDate TEXT NOT NULL DEFAULT (date('now')),
    phone TEXT,
    email TEXT NOT NULL UNIQUE,
    salary REAL NOT NULL CHECK (Salary >= 0),
    supervisorID INTEGER,
    FOREIGN KEY (position) REFERENCES PositionDepartment(position)
    ON UPDATE CASCADE ON DELETE RESTRICT,
    FOREIGN KEY (supervisorID) REFERENCES Employee(employeeID)
    ON UPDATE CASCADE ON DELETE SET NULL
)

Running query in 'sqlite:///library.db'

++
||
++
++

In [8]:
%%sql
CREATE TABLE Item (
    itemID INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    publisher TEXT,
    language TEXT NOT NULL DEFAULT 'English',
    publicationYear INTEGER,
    itemType TEXT NOT NULL CHECK (ItemType IN ('PrintBook','OnlineBook','Magazine','ScientificJournal','Record')),
    availability INTEGER NOT NULL DEFAULT 1 CHECK (Availability IN (0,1))
)

Running query in 'sqlite:///library.db'

++
||
++
++

In [9]:
%%sql
CREATE TABLE ItemShelf (
    itemID INTEGER PRIMARY KEY,
    shelfLocation TEXT NOT NULL,
    FOREIGN KEY (itemID) REFERENCES Item(itemID)
    ON UPDATE CASCADE ON DELETE CASCADE
)

Running query in 'sqlite:///library.db'

++
||
++
++

In [10]:
%%sql
CREATE TABLE PrintBook (
    itemID INTEGER PRIMARY KEY,
    FOREIGN KEY (ItemID) REFERENCES Item(ItemID)
    ON UPDATE CASCADE ON DELETE CASCADE
)

Running query in 'sqlite:///library.db'

++
||
++
++

In [11]:
%%sql
CREATE TABLE OnlineBook (
    itemID INTEGER PRIMARY KEY,
    FOREIGN KEY (ItemID) REFERENCES Item(ItemID)
    ON UPDATE CASCADE ON DELETE CASCADE
)

Running query in 'sqlite:///library.db'

++
||
++
++

In [12]:
%%sql
CREATE TABLE Magazine (
    itemID INTEGER PRIMARY KEY,
    FOREIGN KEY (ItemID) REFERENCES Item(ItemID)
    ON UPDATE CASCADE ON DELETE CASCADE
)

Running query in 'sqlite:///library.db'

++
||
++
++

In [13]:
%%sql
CREATE TABLE ScientificJournal (
    itemID INTEGER PRIMARY KEY,
    FOREIGN KEY (ItemID) REFERENCES Item(ItemID)
    ON UPDATE CASCADE ON DELETE CASCADE
)

Running query in 'sqlite:///library.db'

++
||
++
++

In [14]:
%%sql
CREATE TABLE Record (
    itemID INTEGER PRIMARY KEY,
    FOREIGN KEY (itemID) REFERENCES Item(itemID)
    ON UPDATE CASCADE ON DELETE CASCADE
)

Running query in 'sqlite:///library.db'

++
||
++
++

In [15]:
%%sql
CREATE TABLE Copy (
    itemID INTEGER NOT NULL,
    copyNumber INTEGER NOT NULL,
    condition TEXT NOT NULL DEFAULT 'Good' CHECK (Condition IN ('New','Good','Worn','Damaged')),
    acquisitionDate TEXT NOT NULL DEFAULT (date('now')),
    PRIMARY KEY (itemID, copyNumber),
    FOREIGN KEY (itemID) REFERENCES Item(itemID)
    ON UPDATE CASCADE ON DELETE CASCADE
)

Running query in 'sqlite:///library.db'

++
||
++
++

In [16]:
%%sql
CREATE TABLE Fine (
    fineID INTEGER PRIMARY KEY AUTOINCREMENT,
    loanID INTEGER NOT NULL UNIQUE,
    amount REAL NOT NULL CHECK (amount >= 0),
    paymentStatus TEXT NOT NULL DEFAULT 'Unpaid' CHECK (paymentStatus IN ('Paid','Unpaid')),
    dateAssessed TEXT NOT NULL DEFAULT (date('now')),
    FOREIGN KEY (loanID) REFERENCES Loan(loanID)
    ON UPDATE CASCADE ON DELETE CASCADE
)

Running query in 'sqlite:///library.db'

++
||
++
++

In [17]:
%%sql
CREATE TABLE Loan (
    loanID INTEGER PRIMARY KEY AUTOINCREMENT,
    memberID INTEGER NOT NULL,
    itemID INTEGER NOT NULL,
    copyNumber INTEGER NOT NULL,
    borrowDate TEXT NOT NULL DEFAULT (date('now')),
    dueDate TEXT NOT NULL,
    returnDate TEXT,
    loanStatus TEXT NOT NULL DEFAULT 'Active' CHECK (LoanStatus IN ('Active','Returned','Overdue')),
    FOREIGN KEY (memberID) REFERENCES Member(memberID)
    ON UPDATE CASCADE ON DELETE RESTRICT,
    FOREIGN KEY (itemID, copyNumber) REFERENCES Copy(itemID, copyNumber)
    ON UPDATE CASCADE ON DELETE RESTRICT,
    CHECK (dueDate >= borrowDate),
    CHECK (returnDate IS NULL OR returnDate >= borrowDate)
)

Running query in 'sqlite:///library.db'

++
||
++
++

In [18]:
%%sql
CREATE TABLE Room (
    roomID INTEGER PRIMARY KEY AUTOINCREMENT,
    roomName TEXT NOT NULL UNIQUE,
    floor INTEGER,
    capacity INTEGER NOT NULL CHECK (Capacity > 0)
)

Running query in 'sqlite:///library.db'

++
||
++
++

In [19]:
%%sql
CREATE TABLE Event (
    eventID INTEGER PRIMARY KEY,
    title TEXT NOT NULL,
    eventDate TEXT NOT NULL,
    startTime TEXT NOT NULL,
    endTime TEXT NOT NULL,
    eventType TEXT NOT NULL CHECK (EventType IN
('BookClub','AuthorTalk','ArtShow','FilmScreening','Workshop')),
    roomID INTEGER NOT NULL,
    employeeID INTEGER NOT NULL,
    FOREIGN KEY (roomID) REFERENCES Room(roomID)
    ON UPDATE CASCADE ON DELETE RESTRICT,
    FOREIGN KEY (employeeID) REFERENCES Employee(employeeID)
    ON UPDATE CASCADE ON DELETE RESTRICT,
    CHECK (endTime > startTime),
    UNIQUE (roomID, eventDate, startTime)
)

Running query in 'sqlite:///library.db'

++
||
++
++

In [20]:
%%sql
CREATE TABLE AudienceType (
    audienceTypeID INTEGER PRIMARY KEY AUTOINCREMENT,
    audienceName TEXT NOT NULL UNIQUE
)

Running query in 'sqlite:///library.db'

++
||
++
++

In [21]:
%%sql
CREATE TABLE EventAudience (
    eventID INTEGER NOT NULL,
    audienceTypeID INTEGER NOT NULL,
    PRIMARY KEY (eventID, audienceTypeID),
    FOREIGN KEY (eventID) REFERENCES Event(eventID)
    ON UPDATE CASCADE ON DELETE CASCADE,
    FOREIGN KEY (audienceTypeID) REFERENCES AudienceType(audienceTypeID)
    ON UPDATE CASCADE ON DELETE RESTRICT
)

Running query in 'sqlite:///library.db'

++
||
++
++

In [22]:
%%sql
CREATE TABLE EventRegistration (
    memberID INTEGER NOT NULL,
    eventID INTEGER NOT NULL,
    registrationDate TEXT NOT NULL DEFAULT (date('now')),
    checkedIn INTEGER NOT NULL DEFAULT 0 CHECK (checkedIn IN (0,1)),
    PRIMARY KEY (memberID, eventID),
    FOREIGN KEY (memberID) REFERENCES Member(memberID)
    ON UPDATE CASCADE ON DELETE CASCADE,
    FOREIGN KEY (eventID) REFERENCES Event(eventID)
    ON UPDATE CASCADE ON DELETE CASCADE
)

Running query in 'sqlite:///library.db'

++
||
++
++

In [23]:
%%sql
CREATE TABLE PotentialItem (
    potentialItemID INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    itemType TEXT NOT NULL CHECK (itemType IN
('PrintBook','OnlineBook','Magazine','ScientificJournal','Record')),
vendor TEXT,
estimatedPrice REAL CHECK (estimatedPrice >= 0),
requestDate TEXT NOT NULL DEFAULT (date('now')),
status TEXT NOT NULL DEFAULT 'Requested' CHECK (status IN
('Requested','Approved','Ordered','Received','Rejected'))
)

Running query in 'sqlite:///library.db'

++
||
++
++

Prevent physical copies of OnlineBook items (business rule: online books have no copies)

In [24]:
%%sql
CREATE TRIGGER trg_no_copy_for_online_book
BEFORE INSERT ON Copy
FOR EACH ROW
WHEN (SELECT itemType FROM Item WHERE itemID = NEW.itemID) = 'OnlineBook'
BEGIN
SELECT RAISE(ABORT, 'Online books cannot have physical copies');
END

Running query in 'sqlite:///library.db'

++
||
++
++

an ItemID inserted into a subtype table must match Item.ItemType

In [25]:
%%sql
CREATE TRIGGER trg_check_printbook_type
BEFORE INSERT ON PrintBook
FOR EACH ROW
WHEN (SELECT itemType FROM Item WHERE itemID = NEW.itemID) <> 'PrintBook'
BEGIN
SELECT RAISE(ABORT, 'ItemID does not match ItemType = PrintBook');
END

Running query in 'sqlite:///library.db'

++
||
++
++

In [26]:
%%sql
CREATE TRIGGER trg_check_onlinebook_type
BEFORE INSERT ON OnlineBook
FOR EACH ROW
WHEN (SELECT ItemType FROM item WHERE itemID = NEW.itemID) <> 'OnlineBook'
BEGIN
SELECT RAISE(ABORT, 'ItemID does not match ItemType = OnlineBook');
END

Running query in 'sqlite:///library.db'

++
||
++
++

In [27]:
%%sql
CREATE TRIGGER trg_check_magazine_type
BEFORE INSERT ON Magazine
FOR EACH ROW
WHEN (SELECT ItemType FROM item WHERE itemID = NEW.itemID) <> 'Magazine'
BEGIN
SELECT RAISE(ABORT, 'ItemID does not match ItemType = Magazine');
END

Running query in 'sqlite:///library.db'

++
||
++
++

In [28]:
%%sql
CREATE TRIGGER trg_check_journal_type
BEFORE INSERT ON ScientificJournal
FOR EACH ROW
WHEN (SELECT ItemType FROM item WHERE itemID = NEW.itemID) <> 'ScientificJournal'
BEGIN
SELECT RAISE(ABORT, 'ItemID does not match ItemType = ScientificJournal');
END

Running query in 'sqlite:///library.db'

++
||
++
++

In [29]:
%%sql
CREATE TRIGGER trg_check_record_type
BEFORE INSERT ON Record
FOR EACH ROW
WHEN (SELECT ItemType FROM item WHERE itemID = NEW.itemID) <> 'Record'
BEGIN
SELECT RAISE(ABORT, 'ItemID does not match ItemType = Record');
END

Running query in 'sqlite:///library.db'

++
||
++
++

Prevent a second open loan on the same copy (a copy can only be borrowed by one member at a time)

In [30]:
%%sql
CREATE TRIGGER trg_prevent_double_loan
BEFORE INSERT ON Loan
FOR EACH ROW
WHEN EXISTS (
SELECT 1 FROM Loan
WHERE itemID = NEW.itemID
AND copyNumber = NEW.copyNumber
AND returnDate IS NULL
)
BEGIN
SELECT RAISE(ABORT, 'This copy is already on loan');
END

Running query in 'sqlite:///library.db'

++
||
++
++

Mark item unavailable when a copy is loaned out

In [31]:
%%sql
CREATE TRIGGER trg_item_unavailable_on_loan
AFTER INSERT ON Loan
FOR EACH ROW
WHEN NOT EXISTS (
    SELECT 1 FROM Copy c
    WHERE c.itemID = NEW.itemID
    AND NOT EXISTS (
        SELECT 1 FROM Loan l
        WHERE l.itemID = c.itemID AND l.copyNumber = c.copyNumber AND l.returnDate IS NULL
    )
)
BEGIN
UPDATE Item SET availability = 0 WHERE itemID = NEW.itemID;
END

Running query in 'sqlite:///library.db'

++
||
++
++

When a loan is returned: mark item available again

In [32]:
%%sql
CREATE TRIGGER trg_on_return
AFTER UPDATE OF returnDate ON Loan
FOR EACH ROW
WHEN NEW.returnDate IS NOT NULL
BEGIN
UPDATE Loan SET loanStatus = 'Returned' WHERE loanID = NEW.loanID;

UPDATE Item SET availability = 1
WHERE itemID = NEW.itemID
AND NOT EXISTS (
    SELECT 1 FROM Loan
    WHERE itemID = NEW.itemID AND returnDate IS NULL AND loanID <> NEW.loanID
);

INSERT INTO Fine (loanID, amount, paymentStatus, dateAssessed)
SELECT NEW.loanID,
       ROUND((julianday(NEW.returnDate) - julianday(NEW.dueDate)) * 0.50, 2),
       'Unpaid',
       NEW.returnDate
WHERE NEW.returnDate > NEW.dueDate;
END

Running query in 'sqlite:///library.db'

++
||
++
++

Flag a loan as Overdue automatically if checked while still unreturned past due date

In [33]:
%%sql
DROP TRIGGER IF EXISTS trg_flag_overdue;

Running query in 'sqlite:///library.db'

++
||
++
++

In [34]:
%%sql
CREATE TRIGGER trg_flag_overdue
AFTER UPDATE ON Loan
FOR EACH ROW
WHEN NEW.returnDate IS NULL
AND NEW.dueDate < date('now')
AND NEW.loanStatus = 'Active'
BEGIN
UPDATE Loan SET loanStatus = 'Overdue' WHERE loanID = NEW.loanID;
END

Running query in 'sqlite:///library.db'

++
||
++
++

Prevent removing the last AudienceType link from an Event (every event needs >= 1 audience)

In [35]:
%%sql
CREATE TRIGGER trg_keep_min_one_audience
BEFORE DELETE ON EventAudience
FOR EACH ROW
WHEN (SELECT COUNT(*) FROM EventAudience WHERE eventID = OLD.eventID) <= 1
BEGIN
SELECT RAISE(ABORT, 'Every event must keep at least one audience type');
END

Running query in 'sqlite:///library.db'

++
||
++
++

In [36]:
import sqlite3

conn = sqlite3.connect("library.db")
conn.execute("PRAGMA foreign_keys = ON")
cur = conn.cursor()

print("Connected to library.db")

Connected to library.db


In [37]:
departments = [
    ("Circulation", "Main Floor"),
    ("Cataloging", "Second Floor"),
    ("Programs & Events", "Main Floor"),
    ("Administration", "Third Floor"),  
    ("Information Technology", "Third Floor"),
    ("Human Resources", "Third Floor"),
    ("Acquisitions", "Second Floor"),
    ("Youth Services", "Main Floor"),
    ("Reference Services", "Second Floor"),
    ("Facilities", "Basement"),
]
cur.executemany("INSERT INTO Department (deptName, location) VALUES (?,?)", departments)

In [38]:
postal_codes = [
    ("V3T 1J9", "Surrey", "BC"),
    ("V5K 0A1", "Vancouver", "BC"),
    ("V6B 1A1", "Vancouver", "BC"),
    ("V3S 4N2", "Surrey", "BC"),
    ("V5G 3T4", "Burnaby", "BC"),
    ("V7T 1A1", "West Vancouver", "BC"),
    ("V4N 0A1", "Coquitlam", "BC"),
    ("V2Y 1C7", "Abbotsford", "BC"),
    ("V1M 2G4", "Langley", "BC"),
    ("V9A 1G1", "Victoria", "BC"),
]
cur.executemany("INSERT INTO PostalCode (postalCode, city, province) VALUES (?,?,?)", postal_codes)

In [39]:
positions = [
    ("Circulation Clerk", 1),
    ("Circulation Supervisor", 1),
    ("Cataloging Librarian", 2),
    ("Cataloging Assistant", 2),
    ("Events Coordinator", 3),
    ("Community Librarian", 3),
    ("Library Director", 4),
    ("HR Coordinator", 6),
    ("IT Support Specialist", 5),
    ("Acquisitions Librarian", 7),
]
cur.executemany("INSERT INTO positionDepartment (position, deptID) VALUES (?,?)", positions)

In [40]:
employees = [
    ("Alexis", "Yew", "Library Director", "2015-03-01", "604-555-0101", "a.yew@library.org", 95000, None),
    ("Hala", "Obeid", "Circulation Supervisor", "2018-06-15", "604-555-0102", "h.rahman@library.org", 58000, 1),
    ("Priya", "Nair", "Circulation Clerk", "2021-01-10", "604-555-0103", "p.nair@library.org", 42000, 2),
    ("Marcus", "Chen", "Circulation Clerk", "2022-05-20", "604-555-0104", "m.chen@library.org", 41500, 2),
    ("Sofia", "Reyes", "Cataloging Librarian", "2017-09-11", "604-555-0105", "s.reyes@library.org", 62000, 1),
    ("David", "Kim", "Cataloging Assistant", "2020-02-18", "604-555-0106", "d.kim@library.org", 45000, 5),
    ("Elena", "Popescu", "Events Coordinator", "2019-04-02", "604-555-0107", "e.popescu@library.org", 51000, 1),
    ("Jordan", "Smith", "Community Librarian", "2016-11-23", "604-555-0108", "j.smith@library.org", 60000, 1),
    ("Amara", "Okafor", "IT Support Specialist", "2020-08-30", "604-555-0109", "a.okafor@library.org", 55000, 1),
    ("Liam", "Brennan", "Acquisitions Librarian", "2018-01-14", "604-555-0110", "l.brennan@library.org", 57000, 1),
]
cur.executemany(
    "INSERT INTO Employee (firstName,lastName,position,hireDate,phone,email,salary,supervisorID) VALUES (?,?,?,?,?,?,?,?)",
    employees
)

In [41]:
members = [
    ("Grace", "Tan", "grace.tan@email.com", "778-555-0201", "12 Maple St", "V3T 1J9", "2023-01-15", "Standard"),
    ("Noah", "Patel", "noah.patel@email.com", "778-555-0202", "45 Oak Ave", "V5K 0A1", "2022-06-20", "Student"),
    ("Isabella", "Garcia", "isabella.garcia@email.com", "778-555-0203", "8 Birch Rd", "V6B 1A1", "2021-11-02", "Standard"),
    ("Ethan", "Wong", "ethan.wong@email.com", "778-555-0204", "77 Pine Cres", "V3S 4N2", "2023-03-09", "Senior"),
    ("Mia", "Singh", "mia.singh@email.com", "778-555-0205", "22 Cedar Ln", "V5G 3T4", "2020-08-14", "Standard"),
    ("Lucas", "Ferreira", "lucas.ferreira@email.com", "778-555-0206", "3 Elm Ct", "V7T 1A1", "2024-02-01", "Student"),
    ("Ava", "Kowalski", "ava.kowalski@email.com", "778-555-0207", "91 Willow Dr", "V4N 0A1", "2019-05-30", "Senior"),
    ("Benjamin", "Osei", "benjamin.osei@email.com", "778-555-0208", "14 Aspen Way", "V2Y 1C7", "2023-09-18", "Standard"),
    ("Chloe", "Ivanova", "chloe.ivanova@email.com", "778-555-0209", "60 Fir St", "V1M 2G4", "2022-12-05", "Student"),
    ("Daniel", "Moreau", "daniel.moreau@email.com", "778-555-0210", "5 Spruce Blvd", "V9A 1G1", "2021-07-22", "Standard"),
]
cur.executemany(
    "INSERT INTO Member (firstName,lastName,email,phone,street,postalCode,registrationDate,membershipType) VALUES (?,?,?,?,?,?,?,?)",
    members
)

In [42]:
def add_items(rows, itemType, subtable):
    ids = []
    for (title, publisher, language, year) in rows:
        cur.execute(
            "INSERT INTO Item (title,publisher,language,publicationYear,itemType) VALUES (?,?,?,?,?)",
            (title, publisher, language, year, itemType)
        )
        itemID = cur.lastrowid
        ids.append(itemID)
        cur.execute(f"INSERT INTO {subtable} (itemID) VALUES (?)", (itemID,))
    return ids
 
printbooks = [
    ("The Conjure Woman", "Houghton Mifflin", "English", 1899),
    ("Klara and the Sun", "Faber & Faber", "English", 2021),
    ("Educated", "Random House", "English", 2018),
    ("The Night Circus", "Anchor Books", "English", 2011),
    ("Circe", "Little, Brown", "English", 2018),
    ("The Silent Patient", "Celadon Books", "English", 2019),
    ("Project Hail Mary", "Ballantine", "English", 2021),
    ("The Vanishing Half", "Riverhead Books", "English", 2020),
    ("A Little Life", "Doubleday", "English", 2015),
    ("Piranesi", "Bloomsbury", "English", 2020),
]
onlinebooks = [
    ("Introduction to Algorithms (eBook)", "MIT Press", "English", 2022),
    ("Clean Code (eBook)", "Prentice Hall", "English", 2008),
    ("Sapiens (eBook)", "Harper", "English", 2015),
    ("The Pragmatic Programmer (eBook)", "Addison-Wesley", "English", 2019),
    ("Atomic Habits (eBook)", "Avery", "English", 2018),
    ("Deep Work (eBook)", "Grand Central", "English", 2016),
    ("The Design of Everyday Things (eBook)", "Basic Books", "English", 2013),
    ("Thinking, Fast and Slow (eBook)", "Farrar, Straus and Giroux", "English", 2011),
    ("Dune (eBook)", "Ace Books", "English", 1965),
    ("The Hobbit (eBook)", "HarperCollins", "English", 1937),
]
magazines = [
    ("National Geographic - July Issue", "National Geographic Society", "English", 2026),
    ("The Economist - Weekly Edition", "The Economist Group", "English", 2026),
    ("Time Magazine - Summer Special", "Time USA", "English", 2026),
    ("Wired - Tech Trends", "Conde Nast", "English", 2026),
    ("Scientific Canadian", "SciCan Press", "English", 2026),
    ("Vogue - Fall Preview", "Conde Nast", "English", 2025),
    ("The New Yorker - Fiction Issue", "Conde Nast", "English", 2025),
    ("Popular Mechanics", "Hearst", "English", 2026),
    ("Canadian Geographic", "RCGS", "English", 2026),
    ("BC Living", "Living Media", "English", 2026),
]
journals = [
    ("Journal of Machine Learning Research", "JMLR Inc.", "English", 2025),
    ("Nature Communications", "Springer Nature", "English", 2025),
    ("Canadian Journal of Computer Science", "CJCS Press", "English", 2024),
    ("Journal of Library Science", "ALA Press", "English", 2023),
    ("IEEE Transactions on AI", "IEEE", "English", 2025),
    ("The Lancet Digital Health", "Elsevier", "English", 2025),
    ("Journal of Applied Linguistics", "Cambridge UP", "English", 2022),
    ("ACM Computing Surveys", "ACM", "English", 2024),
    ("Canadian Medical Association Journal", "CMA", "English", 2025),
    ("Journal of Environmental Studies", "UBC Press", "English", 2023),
]
records = [
    ("Kind of Blue", "Columbia Records", "English", 1959),
    ("Rumours", "Warner Bros", "English", 1977),
    ("Abbey Road", "Apple Records", "English", 1969),
    ("Thriller", "Epic Records", "English", 1982),
    ("Blue Train", "Blue Note", "English", 1957),
    ("Back to Black", "Island Records", "English", 2006),
    ("Random Access Memories", "Columbia Records", "English", 2013),
    ("Blonde", "Boo Boo Records", "English", 2016),
    ("Folklore", "Republic Records", "English", 2020),
    ("Currents", "Interscope", "English", 2015),
]
 
printbook_ids = add_items(printbooks, "PrintBook", "PrintBook")
onlinebook_ids = add_items(onlinebooks, "OnlineBook", "OnlineBook")
magazine_ids = add_items(magazines, "Magazine", "Magazine")
journal_ids = add_items(journals, "ScientificJournal", "ScientificJournal")
record_ids = add_items(records, "Record", "Record")
 
physical_ids = printbook_ids + magazine_ids + journal_ids + record_ids  # excludes OnlineBook
shelves = ["A1-01","A1-02","A2-01","A2-02","B1-01","B1-02","B2-01","B2-02",
           "C1-01","C1-02","C2-01","C2-02","D1-01","D1-02","D2-01","D2-02",
           "E1-01","E1-02","E2-01","E2-02","F1-01","F1-02","F2-01","F2-02",
           "G1-01","G1-02","G2-01","G2-02","H1-01","H1-02","H2-01","H2-02",
           "J1-01","J1-02","J2-01","J2-02","K1-01","K1-02","K2-01","K2-02"]
for itemID, shelf in zip(physical_ids, shelves):
    cur.execute("INSERT INTO ItemShelf (itemID, shelfLocation) VALUES (?,?)", (itemID, shelf))

In [43]:
for itemID in physical_ids:
    for copyNum in (1, 2):
        cur.execute(
            "INSERT INTO Copy (itemID, copyNumber, condition) VALUES (?,?,?)",
            (itemID, copyNum, "Good")
        )
 

rooms = [
    ("Conference Room A", 1, 20),
    ("Conference Room B", 1, 20),
    ("Story Time Room", 1, 15),
    ("Art Studio", 2, 12),
    ("Community Hall", 1, 80),
    ("Teen Lounge", 2, 25),
    ("Board Room", 3, 10),
    ("Computer Lab", 2, 18),
    ("Reading Garden", 1, 30),
    ("Multipurpose Room", 2, 40),
]
cur.executemany("INSERT INTO Room (roomName, floor, capacity) VALUES (?,?,?)", rooms)

In [44]:
events = [
    ("Mystery Book Club", "2026-08-05", "18:00", "19:30", "BookClub", 1, 8),
    ("Meet the Author: Sofia Reyes", "2026-08-08", "17:00", "18:30", "AuthorTalk", 5, 8),
    ("Local Artists Showcase", "2026-08-10", "13:00", "16:00", "ArtShow", 4, 7),
    ("Classic Film Night: Casablanca", "2026-08-12", "19:00", "21:00", "FilmScreening", 5, 7),
    ("Intro to Coding Workshop", "2026-08-14", "10:00", "12:00", "Workshop", 8, 9),
    ("Teen Sci-Fi Book Club", "2026-08-16", "16:00", "17:30", "BookClub", 6, 8),
    ("Resume Writing Workshop", "2026-08-18", "14:00", "16:00", "Workshop", 7, 7),
    ("Documentary Screening Night", "2026-08-20", "19:00", "21:00", "FilmScreening", 5, 7),
    ("Watercolor Painting Class", "2026-08-22", "13:00", "15:00", "Workshop", 4, 7),
    ("Author Panel: Local Voices", "2026-08-25", "17:30", "19:00", "AuthorTalk", 5, 8),
]
cur.executemany(
    "INSERT INTO Event (title,eventDate,startTime,endTime,eventType,roomID,employeeID) VALUES (?,?,?,?,?,?,?)",
    events
)

In [45]:
audiences = ["Children","Teens","Adults","Seniors","Toddlers","Young Adults",
             "Families","New Immigrants","Job Seekers","ESL Learners"]
cur.executemany("INSERT INTO AudienceType (audienceName) VALUES (?)", [(a,) for a in audiences])

In [46]:
event_audience_map = [
    (1, [3]), (2, [3,4]), (3, [3,6]), (4, [3,4]), (5, [6,9]),
    (6, [2,6]), (7, [3,6,9]), (8, [3]), (9, [3,4]), (10, [3,4,6]),
]
for eventID, audienceIDs in event_audience_map:
    for aID in audienceIDs:
        cur.execute("INSERT INTO EventAudience (eventID, audienceTypeID) VALUES (?,?)", (eventID, aID))

In [47]:
registrations = [
    (1,1,0), (1,2,1), (2,3,0), (2,4,0), (3,5,1),
    (4,6,0), (5,7,0), (6,8,0), (7,9,1), (8,10,0),
    (9,1,0), (10,2,0),
]
for memberID, eventID, checkedIn in registrations:
    cur.execute(
        "INSERT INTO EventRegistration (memberID, eventID, checkedIn) VALUES (?,?,?)",
        (memberID, eventID, checkedIn)
    )
 
conn.commit()

In [48]:
loans = [
    (1, printbook_ids[0], 1, "2026-07-01", "2026-07-15", "2026-07-14"),   # returned on time
    (2, printbook_ids[1], 1, "2026-07-05", "2026-07-19", "2026-07-25"),  # returned LATE
    (3, printbook_ids[2], 1, "2026-07-10", "2026-07-24", None),          # still active
    (4, magazine_ids[0], 1, "2026-07-02", "2026-07-16", "2026-07-16"),   # returned on time
    (5, magazine_ids[1], 1, "2026-06-20", "2026-07-04", "2026-07-12"),   # returned LATE
    (6, journal_ids[0], 1, "2026-07-12", "2026-07-26", None),            # still active
    (7, journal_ids[1], 1, "2026-06-15", "2026-06-29", "2026-07-06"),    # returned LATE
    (8, record_ids[0], 1, "2026-07-08", "2026-07-22", "2026-07-21"),     # returned on time
    (9, record_ids[1], 1, "2026-06-25", "2026-07-09", None),             # still active (now overdue)
    (10, printbook_ids[3], 1, "2026-07-14", "2026-07-28", None),         # still active
    (1, printbook_ids[4], 1, "2026-07-18", "2026-08-01", None),          # still active
    (2, magazine_ids[2], 1, "2026-06-10", "2026-06-24", "2026-07-02"),   # returned LATE
]
for memberID, itemID, copyNumber, borrowDate, dueDate, returnDate in loans:
    cur.execute(
        "INSERT INTO Loan (memberID,itemID,copyNumber,borrowDate,dueDate) VALUES (?,?,?,?,?)",
        (memberID, itemID, copyNumber, borrowDate, dueDate)
    )
    loanID = cur.lastrowid
    if returnDate:
        # UPDATE (not INSERT) so trg_on_return fires and auto-creates a Fine if late
        cur.execute("UPDATE Loan SET returnDate = ? WHERE loanID = ?", (returnDate, loanID))
 
conn.commit()

In [49]:
cur.execute("""
    UPDATE Loan SET loanStatus = 'Overdue'
    WHERE returnDate IS NULL AND dueDate < date('now')
""")
 
cur.execute("SELECT fineID FROM Fine ORDER BY fineID LIMIT 1")
row = cur.fetchone()
if row:
    cur.execute("UPDATE Fine SET paymentStatus = 'Paid' WHERE fineID = ?", (row[0],))
 
conn.commit()

In [50]:
historical_late_loans = [
    (3, journal_ids[2], 1, "2026-05-01", "2026-05-15", "2026-05-20", 2.50, "Paid"),
    (4, printbook_ids[5], 1, "2026-05-03", "2026-05-17", "2026-05-19", 1.00, "Paid"),
    (5, magazine_ids[3], 1, "2026-05-10", "2026-05-24", "2026-06-01", 3.50, "Unpaid"),
    (6, record_ids[2], 1, "2026-05-12", "2026-05-26", "2026-06-05", 4.50, "Paid"),
    (7, printbook_ids[6], 1, "2026-05-15", "2026-05-29", "2026-06-03", 2.50, "Unpaid"),
    (8, journal_ids[3], 1, "2026-05-18", "2026-06-01", "2026-06-04", 1.50, "Paid"),
]
for memberID, itemID, copyNumber, borrowDate, dueDate, returnDate, fineAmount, payStatus in historical_late_loans:
    cur.execute(
        "INSERT INTO Loan (memberID,itemID,copyNumber,borrowDate,dueDate,returnDate,loanStatus) "
        "VALUES (?,?,?,?,?,?,'Returned')",
        (memberID, itemID, copyNumber, borrowDate, dueDate, returnDate)
    )
    loanID = cur.lastrowid
    cur.execute(
        "INSERT INTO Fine (loanID, amount, paymentStatus, dateAssessed) VALUES (?,?,?,?)",
        (loanID, fineAmount, payStatus, returnDate)
    )
 
conn.commit()

In [51]:
potential_items = [
    ("The Fraud", "PrintBook", "Ingram", 24.99, "2026-07-01", "Requested"),
    ("Fourth Wing (eBook)", "OnlineBook", "OverDrive", 12.99, "2026-07-02", "Approved"),
    ("Popular Science - Fall Issue", "Magazine", "Curtis Circulation", 6.99, "2026-07-03", "Requested"),
    ("Journal of Data Science", "ScientificJournal", "SAGE", 199.00, "2026-07-04", "Ordered"),
    ("Midnights (Vinyl)", "Record", "Republic Records", 34.99, "2026-07-05", "Requested"),
    ("Tomorrow, and Tomorrow, and Tomorrow", "PrintBook", "Ingram", 22.50, "2026-07-06", "Received"),
    ("The Creative Act (eBook)", "OnlineBook", "OverDrive", 14.99, "2026-07-07", "Requested"),
    ("Bon Appetit - Holiday Issue", "Magazine", "Curtis Circulation", 5.99, "2026-07-08", "Rejected"),
    ("Nature Neuroscience", "ScientificJournal", "Springer", 220.00, "2026-07-09", "Requested"),
    ("Un Verano Sin Ti (Vinyl)", "Record", "Rimas Entertainment", 29.99, "2026-07-10", "Approved"),
]
cur.executemany(
    "INSERT INTO PotentialItem (title,itemType,vendor,estimatedPrice,requestDate,status) VALUES (?,?,?,?,?,?)",
    potential_items
)
 
conn.commit()

In [52]:
tables = ["Department","PostalCode","Member","positionDepartment","Employee","Item",
          "PrintBook","OnlineBook","Magazine","ScientificJournal","Record","ItemShelf",
          "Copy","Loan","Fine","Room","Event","AudienceType","EventAudience",
          "EventRegistration","PotentialItem"]
print(f"{'Table':<20}{'Rows':>6}")
for t in tables:
    n = cur.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    flag = "" if n >= 10 else "  <-- UNDER 10"
    print(f"{t:<20}{n:>6}{flag}")
 
conn.close()

Table                 Rows
Department              10
PostalCode              10
Member                  10
positionDepartment      10
Employee                10
Item                    50
PrintBook               10
OnlineBook              10
Magazine                10
ScientificJournal       10
Record                  10
ItemShelf               40
Copy                    80
Loan                    18
Fine                    10
Room                    10
Event                   10
AudienceType            10
EventAudience           20
EventRegistration       12
PotentialItem           10


In [53]:
# Find an item in the library
def find_item(title):
    conn = sqlite3.connect('library.db')
    cursor = conn.cursor()

    query = 'SELECT itemID, title, publisher, language, publicationYear, itemType, availability FROM item WHERE title LIKE ?'
    search = f"%{title}%"
    try:
        cursor.execute(query, (search, ))
        results = cursor.fetchall()
        if not results:
            print(f"No matching records.")
            return []
        else:
            for row in results:
                print(row)
        return results
    except sqlite3.Error as e:
        print(f"Error: {e}")
        return []
    finally:
        conn.close()

In [54]:
# Borrow an item from the library
def borrow_item(memberID, itemID):
    conn = sqlite3.connect('library.db')
    cursor = conn.cursor()

    query = """SELECT copyNumber from Copy
                WHERE itemID = ?
                AND copyNumber NOT IN (SELECT copyNumber FROM Loan WHERE itemID = ? AND returnDate IS NULL)
                ORDER BY copyNumber
                LIMIT 1"""
    try:
        cursor.execute(query, (itemID, itemID))
        results = cursor.fetchone()

        if not results:
            print("No copies available.")
            return []

        copyNumber = results[0]
        cursor.execute(
            "INSERT INTO Loan (memberID, itemID, copyNumber, dueDate) "
            "VALUES (?, ?, ?, date('now', '+14 days'))",
            (memberID, itemID, copyNumber)
        )
        conn.commit()
        loanID = cursor.lastrowid
        print(f"Borrowed item {itemID} (copy {copyNumber}) for member {memberID}. "
              f"Loan ID {loanID}, due in 14 days.")
        return loanID

    except sqlite3.IntegrityError as e:
        print(f"Could not complete loan: {e}")
        return []
    except sqlite3.Error as e:
        print(f"Error: {e}")
        return []
    finally:
        conn.close()

In [55]:
# Return borrowed item
def return_item(loanID):
    conn = sqlite3.connect('library.db')
    cursor = conn.cursor()

    query = 'SELECT itemID, returnDate FROM Loan WHERE loanID = ?'
    try:
        cursor.execute(query, (loanID, ))
        results = cursor.fetchone()

        if not results:
            print("No loan found.")
            return []

        itemID, returnDate = results
        if returnDate is not None:
            print(f"Loan {loanID} was already returned on {returnDate}.")
            return []

        cursor.execute(
            "UPDATE Loan SET returnDate = date('now') WHERE loanID = ?",
            (loanID,)
        )
        conn.commit()

        cursor.execute("SELECT amount FROM Fine WHERE loanID = ?", (loanID,))
        fine = cursor.fetchone()

        if fine:
            print(f"Loan {loanID} returned late. Fine assessed: ${fine[0]:.2f}")
        else:
            print(f"Loan {loanID} returned on time. No fine.")

        return loanID

    except sqlite3.Error as e:
        print(f"Error: {e}")
        return []
    finally:
        conn.close()

In [56]:
# Donate an item to the library
def donate_item(title, publisher, language, publicationYear, itemType):
    conn = sqlite3.connect('library.db')
    cursor = conn.cursor()

    valid_types = ["PrintBook", "OnlineBook", "Magazine", "ScientificJournal", "Record"]
    if itemType not in valid_types:
        print(f"Invalid itemType. Must be: {valid_types}")
        conn.close()
        return []

    try:
        cursor.execute(
            "INSERT INTO Item (title, publisher, language, publicationYear, itemType) "
            "VALUES (?, ?, ?, ?, ?)",
            (title, publisher, language, publicationYear, itemType)
        )
        itemID = cursor.lastrowid
        cursor.execute(f"INSERT INTO {itemType} (itemID) VALUES (?)", (itemID,))
        if itemType != "OnlineBook":
            cursor.execute(
                "INSERT INTO Copy (itemID, copyNumber, condition) VALUES (?, 1, 'New')",
                (itemID,)
            )
        conn.commit()
        print(f"Donated '{title}' as {itemType}. New itemID: {itemID}.")
        return itemID

    except sqlite3.IntegrityError as e:
        conn.rollback()
        print(f"Could not complete donation: {e}")
        return []
    except sqlite3.Error as e:
        conn.rollback()
        print(f"Error: {e}")
        return []
    finally:
        conn.close()

In [57]:
# Find an event in the library
def find_event(keyword):
    conn = sqlite3.connect('library.db')
    cursor = conn.cursor()
    query = """SELECT eventID, title, eventDate, startTime, endTime, eventType, roomID
               FROM Event WHERE title LIKE ? ORDER BY eventDate, startTime"""
    search = f"%{keyword}%"
    try:
        cursor.execute(query, (search,))
        results = cursor.fetchall()
        if not results:
            print("No matching events.")
            return []
        for row in results:
            print(row)
        return results
    except sqlite3.Error as e:
        print(f"Error: {e}")
        return []
    finally:
        conn.close()
 

In [58]:
# Register for an event in the library
def register_for_event(memberID, eventID):
    conn = sqlite3.connect('library.db')
    cursor = conn.cursor()
    try:
        cursor.execute(
            """SELECT r.capacity FROM Event e
               JOIN Room r ON e.roomID = r.roomID
               WHERE e.eventID = ?""",
            (eventID,)
        )
        room = cursor.fetchone()
        if not room:
            print("No such event.")
            return []
        capacity = room[0]
 
        cursor.execute(
            "SELECT COUNT(*) FROM EventRegistration WHERE eventID = ?",
            (eventID,)
        )
        current = cursor.fetchone()[0]
        if current >= capacity:
            print("This event is full.")
            return []
 
        cursor.execute(
            "INSERT INTO EventRegistration (memberID, eventID) VALUES (?, ?)",
            (memberID, eventID)
        )
        conn.commit()
        print(f"Member {memberID} registered for event {eventID}.")
        return (memberID, eventID)
    except sqlite3.IntegrityError as e:
        print(f"Could not complete registration: {e}")
        return []
    except sqlite3.Error as e:
        print(f"Error: {e}")
        return []
    finally:
        conn.close()

In [59]:
# Volunteer for the library
def volunteer_for_library(memberID, interestArea, availability):
    conn = sqlite3.connect('library.db')
    cursor = conn.cursor()
    try:
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS Volunteer (
                volunteerID INTEGER PRIMARY KEY AUTOINCREMENT,
                memberID INTEGER NOT NULL,
                interestArea TEXT,
                availability TEXT,
                applicationDate TEXT NOT NULL DEFAULT (date('now')),
                status TEXT NOT NULL DEFAULT 'Pending'
                    CHECK (status IN ('Pending','Approved','Rejected')),
                FOREIGN KEY (memberID) REFERENCES Member(memberID)
                ON UPDATE CASCADE ON DELETE CASCADE
            )
        """)
        cursor.execute(
            "INSERT INTO Volunteer (memberID, interestArea, availability) VALUES (?, ?, ?)",
            (memberID, interestArea, availability)
        )
        conn.commit()
        volunteerID = cursor.lastrowid
        print(f"Volunteer application submitted (ID {volunteerID}) for member {memberID}.")
        return volunteerID
    except sqlite3.IntegrityError as e:
        print(f"Could not submit volunteer application: {e}")
        return []
    except sqlite3.Error as e:
        print(f"Error: {e}")
        return []
    finally:
        conn.close()
 

In [61]:
# Ask for help from a librarian
def ask_librarian():
    conn = sqlite3.connect('library.db')
    cursor = conn.cursor()
    query = """SELECT firstName, lastName, email, phone, position
               FROM Employee WHERE position LIKE '%Librarian%'"""
    try:
        cursor.execute(query)
        results = cursor.fetchall()
        if not results:
            print("No librarians are currently on staff.")
            return []
        print("You can reach one of the following librarians for help:")
        for row in results:
            print(row)
        return results
    except sqlite3.Error as e:
        print(f"Error: {e}")
        return []
    finally:
        conn.close()
 